In [1]:
import os
os.environ["HOME"] = "/mnt/nas/shuvranshu"
os.environ["HF_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["XDG_CACHE_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"

os.makedirs("/mnt/nas/shuvranshu/huggingface_cache", exist_ok=True)

In [2]:
from langchain_community.document_loaders import   DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import numpy as np
from dotenv import load_dotenv

#hf token 
load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
loader= DirectoryLoader('documents',glob='*.pdf',loader_cls=PyPDFLoader)
docs=loader.lazy_load()

In [16]:
#KG implementation
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class Triple:
    subj: str
    rel: str
    obj: str

class SimpleKG:
    def __init__(self):
        self.triples: List[Triple] = []

    def add_triple(self, subj: str, rel: str, obj: str):
        self.triples.append(Triple(subj, rel, obj))

    def find_triples(self, entity: str) -> List[Triple]:
        # return all triples where entity is subject or object
        return [t for t in self.triples if t.subj == entity or t.obj == entity]


KG = SimpleKG()
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_triples_spacy(text):
    doc = nlp(text)
    triples = []
    for token in doc:
        if token.dep_ in ("ROOT", "relcl"):  # verbs or relations
            subj = [w.text for w in token.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w.text for w in token.rights if w.dep_ in ("dobj", "pobj", "attr")]
            if subj and obj:
                triples.append((" ".join(subj), token.lemma_, " ".join(obj)))
    return triples




#link entities in query to KG entities
import spacy
nlp = spacy.load("en_core_web_sm")
def extract_entity_mentions(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents] or [chunk.text for chunk in doc.noun_chunks]

def link_entities(query, kg_entities):
    # simple substring + optional embedding similarity
    mentions = extract_entity_mentions(query)  
    entity_map = {}
    for m in mentions:
        entity_map[m] = [e for e in kg_entities if m.lower() in e.lower()]
    return entity_map

def retrieve_kg_context(query, KG: SimpleKG):
    kg_entities = list(set([t.subj for t in KG.triples] + [t.obj for t in KG.triples]))
    entity_map = link_entities(query, kg_entities)
    triples_text = []
    for ents in entity_map.values():
        for ent in ents:
            for t in KG.find_triples(ent):
                triples_text.append(f"{t.subj} {t.rel} {t.obj}")
    return "\n".join(triples_text)


#combine kg retrieval and context retrieval
def get_combined_context(query, retriever, KG):
    # 1. Retrieve text from FAISS DB
    text_docs = retriever.invoke(query)
    text_context = "\n\n".join([d.page_content for d in text_docs])
    #print(f"context:{text_context}")
    if(len(text_docs)==0):
        return None
    
    # 2. Retrieve KG triples
    kg_context = retrieve_kg_context(query, KG)

    # 3. Combine for final LLM input
    combined_context = f"KG Facts:\n{kg_context}\n\nTextual Context:\n{text_context}"
    return combined_context

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     
    chunk_overlap=100,   
    separators=["\n\n", "\n", " ", ""]
)
chunks = []
for doc in docs:
    triples = extract_triples_spacy(doc.page_content)
    for (subj, rel, obj) in triples:
        KG.add_triple(subj.strip(), rel.strip(), obj.strip())
    chunks.extend(
        text_splitter.split_documents([doc]) 
    )
print(len(chunks))



7


In [7]:
print(chunks[3].page_content[:])

intended for inhalation without combustion 
 
(2) The amount of applicable tax referred to in sub-rule (1) shall be determined in the following manner, 
namely: —


In [8]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   model_kwargs={"device": "cuda:2"},
                                   encode_kwargs={"normalize_embeddings": True}
                                   )

/tmp/ipykernel_3343274/1272004119.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
#FAISS cosine similarity index
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

dimension = 384


index = faiss.IndexFlatIP(dimension)

In [10]:
vectorstore = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore({}), 
    index_to_docstore_id={}
)

vectorstore.add_documents(chunks)
vectorstore.save_local("faiss_index")


In [11]:

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
    )

In [12]:
llm = HuggingFacePipeline.from_model_id(
    # model_id="/mnt/nas/shuvranshu/huggingface_cache/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b", 
    model_id="meta-llama/Llama-3.1-8B",
    # model_id="meta-llama/Llama-3.2-3B-Instruct",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 256,
        "do_sample": False,
    },
    device=1
)


Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.56s/it]


In [13]:

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
1.You are a factual assistant. Use the following context to answer the question.
2.Do NOT add information that is not supported by the context.
if you don't find the answer in the context then simply say NO.
3.Do not use your own knowledge.
Context:
{context}

Question: {question}
Answer:
"""
)
chain = prompt | llm


In [23]:
question="According to the notification, how is the 'retail sale price' determined in complex scenarios, such as when multiple prices are listed or prices are changed?"
context = get_combined_context(question,retriever, KG)
response = chain.invoke({
        "context":context,
        "question":question,
        }
        )
print(f"answer:{response.split('Answer:')[-1].strip()}")    


/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


answer:Explanation. — For the purposes of this rule, — 
(a) “applicable tax” means IGST or CGST or SGST or UTGST as the case may be. 
(b) "retail sale price" means the maximum price declared on goods at which such goods in packaged 
form may be sold to the ultimate consumer and includes all taxes, duties, surcharge or cess by 
whatever name called; 
(c) where on the package of any specified goods more than one retail sale price is declared, the 
maximum of such retail sale price shall be deemed to be the retail sale price; 
(d) where the retail sale price declared on packages of any specified goods is altered to increase the 
retail sale price, the retail sale price shall be deemed to be the retail sale price declared on such 
goods, less the amount of tax as applicable; 
(e) where different retail sale prices are declared on different packages for the sale of any specified 
goods above in packaged form in different areas, each such retail sale price shall be the retail sale 
price for